# DeepSeek-OCR × HLLSet Cortex — Pipeline Validation

> **Notebook 01** — July 28, 2026  
> **Prerequisites:** `hllset-py` built and installed
> ```bash
> cd crates/hllset_py && maturin develop --release
> ```

Validates the HLLSet-based semantic compressor for DeepSeek-OCR.
Built on **hllset-next** per STANDARD.md — reference implementation.

## Architecture

```
Vision Encoder → OCR text
  → DocumentTokenizer (words + bigrams + trigrams + sentence hashes)
    → MurmurHash3 → HLLSet (32,768-bit bitmap)
      → ∩ gate_TF HLLSet (decoder vocabulary filter)
        → TokenLut (monotonic TF, pre-gate)
          → materialize (TF-ranked disambiguation)
            → BPE encode → token IDs → Decoder
```

## Tests
1. DocumentTokenizer — 3-gram structural encoding
2. HLLSet — IICA properties, content-addressing
3. gate_TF HLLSet — vocabulary as content-addressed gate
4. HLLSetFilter — persistent LUT, TF-ranked materialization
5. Multi-document learning — TF accumulation, convergence
6. Cross-document similarity — BSS correlation matrix
7. OCRPipeline — gate + filter + BPE encode
8. Latent vocabulary — TF survives gate changes

In [1]:
import sys
from pathlib import Path

# ── Try direct import (pip install -e . or pip install hllset-cortex) ──
try:
    import hllset_py
    from hllset_cortex import (
        DocumentTokenizer, HLLSetFilter, FilterResult, FilterStats,
        OCRPipeline, PipelineResult, GateInfo,
    )
    print(f"hllset_py:  {[x for x in dir(hllset_py) if not x.startswith('_')]}")
    print("hllset_cortex ready (via installed package)")
except ModuleNotFoundError:
    # ── Fallback: auto-detect project root and add to path ──────────────
    _nb_dir = Path.cwd()
    _root = _nb_dir
    while _root != _root.parent:
        if (_root / "crates" / "hllset_py" / "Cargo.toml").exists():
            break
        _root = _root.parent
    else:
        _root = _nb_dir
    _root = _root.resolve()
    sys.path.insert(0, str(_root.parent))

    # Search for hllset_py in .venv and conda site-packages
    _candidates = []
    for _base in [_root / ".venv", Path.home() / ".conda" / "envs" / "deepseek-ocr"]:
        for _d in (_base / "lib").glob("python3*/site-packages"):
            _candidates.append(str(_d))
    for _p in _candidates:
        if _p not in sys.path:
            sys.path.insert(0, _p)

    try:
        import hllset_py
        from hllset_cortex import (
            DocumentTokenizer, HLLSetFilter, FilterResult, FilterStats,
            OCRPipeline, PipelineResult, GateInfo,
        )
        print(f"hllset_py:  {[x for x in dir(hllset_py) if not x.startswith('_')]}")
        print("hllset_cortex ready (via path detection)")
    except ModuleNotFoundError:
        print("ERROR: hllset_py not found. Build it first:")
        print(f"  cd {_root}/crates/hllset_py")
        print(f"  maturin build --release")
        print(f"  pip install target/wheels/hllset_py-*.whl")
        print(f"  pip install -e {_root}")
        raise


hllset_py:  ['HLLSet', 'TokenLut', 'hllset_py', 'materialize', 'materialize_top_n', 'murmur3_hash_py', 'token_to_position_py']
hllset_cortex ready (via installed package)


---
## 1. DocumentTokenizer — 3-Gram Structural Encoding

Produces: words (1-gram) + bigrams + trigrams + sentence hashes.
The 3-gram encoding makes the gate intersection effectively deterministic
(false positive ~10⁻⁵). No gate filtering at the tokenizer level —
that happens at the HLLSet lattice level.

In [2]:
tok = DocumentTokenizer(max_tokens=4096)

sample = """The neural network model processes image data for object detection.
Deep learning has revolutionized computer vision and pattern recognition.
Natural language processing enables machines to understand human text."""

tokens = tok.tokenize(sample)
words_only = tok.tokenize_words_only(sample)

words   = [t for t in tokens if '::' not in t and not t.startswith('sentence_')]
bigrams = [t for t in tokens if t.count('::') == 1]
trigrams = [t for t in tokens if t.count('::') == 2]
sentence_h = [t for t in tokens if t.startswith('sentence_')]

print(f'Words only:      {len(words_only):3d}  {words_only}')
print(f'1-grams (words): {len(words):3d}  {words}')
print(f'2-grams:         {len(bigrams):3d}  {bigrams[:4]}...')
print(f'3-grams:         {len(trigrams):3d}  {trigrams[:3]}...')
print(f'Sentence hashes: {len(sentence_h):3d}  {sentence_h}')
print(f'Total tokens:    {len(tokens):3d}')

Words only:       27  ['the', 'neural', 'network', 'model', 'processes', 'image', 'data', 'for', 'object', 'detection', 'deep', 'learning', 'has', 'revolutionized', 'computer', 'vision', 'and', 'pattern', 'recognition', 'natural', 'language', 'processing', 'enables', 'machines', 'understand', 'human', 'text']
1-grams (words):  27  ['the', 'neural', 'network', 'model', 'processes', 'image', 'data', 'for', 'object', 'detection', 'deep', 'learning', 'has', 'revolutionized', 'computer', 'vision', 'and', 'pattern', 'recognition', 'natural', 'language', 'processing', 'enables', 'machines', 'understand', 'human', 'text']
2-grams:          26  ['the::neural', 'neural::network', 'network::model', 'model::processes']...
3-grams:          25  ['the::neural::network', 'neural::network::model', 'network::model::processes']...
Sentence hashes:   3  ['sentence_51cc162e', 'sentence_6060ef74', 'sentence_3b0ed466']
Total tokens:     81


---
## 2. HLLSet — IICA Properties

Idempotent: same tokens → same HLLSet, every time.  
Immutable: once created, never changes.  
Content-Addressed: key = SHA1 of serialized bytes.

In [3]:
hllset = hllset_py.HLLSet.from_tokens(tokens)
print(f'HLLSet popcount:    {hllset.popcount()}')
print(f'Cardinality:        {hllset.cardinality():.1f}')
print(f'Content key:        {hllset.content_key()[:48]}...')
print(f'Active positions:   {len(hllset.active_positions())} bits')
print(f'Non-zero registers: {hllset.non_zero_registers()}/1024')

# IICA: idempotence + content-addressability
h2 = hllset_py.HLLSet.from_tokens(tokens)
assert hllset.popcount() == h2.popcount()
assert hllset.content_key() == h2.content_key()
print(f'\nIICA verified: same tokens → same key')

h3 = hllset_py.HLLSet.from_tokens(['different', 'content'])
assert hllset.content_key() != h3.content_key()
print(f'Different tokens → different key')

HLLSet popcount:    79
Cardinality:        80.0
Content key:        h:3dd01f2a2248ed166df742afcccdef76f1308ca6...
Active positions:   79 bits
Non-zero registers: 78/1024

IICA verified: same tokens → same key
Different tokens → different key


---
## 3. gate_TF HLLSet — Vocabulary as Content-Addressed Gate

The decoder's BPE vocabulary is ingested as a **gate_TF HLLSet** —
content-addressed, immutable, system-global. Documents are intersected
with it to filter invalid bit positions at the lattice level.

This is **probabilistic filtering**: an invalid word survives only if
all its hash positions collide with valid vocabulary words. With 3-gram
encoding, probability ~10⁻⁵ — effectively deterministic.

In [4]:
bpe_vocab = sorted(set([
    'the', 'neural', 'network', 'model', 'processes', 'image', 'data',
    'object', 'detection', 'deep', 'learning', 'revolutionized',
    'computer', 'vision', 'pattern', 'recognition', 'natural',
    'language', 'processing', 'enables', 'machines', 'understand',
    'human', 'text', 'training', 'classification', 'feature',
    'extraction', 'algorithm', 'analysis', 'document', 'system',
]))

gate_hllset = hllset_py.HLLSet.from_tokens(bpe_vocab)
print(f'Vocabulary:   {len(bpe_vocab)} words')
print(f'Gate popcount: {gate_hllset.popcount()}')
print(f'Gate key:      {gate_hllset.content_key()[:48]}...')

# Intersection: document ∩ gate
filtered = hllset.intersection(gate_hllset)
pct = 100 - filtered.popcount() * 100 // max(hllset.popcount(), 1)
print(f'\nDoc bits:      {hllset.popcount()}')
print(f'After gate ∩:  {filtered.popcount()}  ({pct}% filtered)')

# Gate is idempotent
gate2 = hllset_py.HLLSet.from_tokens(bpe_vocab)
assert gate_hllset.content_key() == gate2.content_key()
print(f'Gate IICA: same vocab → same gate key')

Vocabulary:   32 words
Gate popcount: 32
Gate key:      h:5fcbe6f0e587de891921f412c96e099a3e6ae03b...

Doc bits:      79
After gate ∩:  24  (70% filtered)
Gate IICA: same vocab → same gate key


---
## 4. HLLSetFilter — LUT + TF-Ranked Materialization

Orchestrates: tokenize → HLLSet → gate ∩ → LUT → materialize.
LUT is a persistent singleton with monotonic TF (CRDT).
Per STANDARD.md Appendix D: TF earned through experience.

In [5]:
filt = HLLSetFilter(max_tokens=4096)
filt.gate_hllset = gate_hllset

print(f'LUT (cold start): {filt.lut.len()} tokens')

text = 'The neural quantum blockchain processes image data for object detection.'
result = filt.process(text)

print(f'\nDocument: {text}')
print(f'  Input words:         {result.stats.input_tokens}')
print(f'  HLLSet bits (all):   {result.stats.hllset_popcount}')
print(f'  After gate ∩:        {result.stats.gate_popcount}')
print(f'  Bits filtered:       {result.stats.hllset_popcount - result.stats.gate_popcount}')
print(f'  Output tokens:       {result.stats.output_tokens}')
print(f'  Materialized:        {result.tokens}')
print(f'  LUT after 1 doc:     {result.lut_size}')

comp = result.compare_to(text)
print(f'  Jaccard:             {comp.jaccard:.3f}')
print(f'  Lost (filtered):     {comp.lost_words}')
print(f'  False positives:     {comp.novel_words}')

LUT (cold start): 0 tokens

Document: The neural quantum blockchain processes image data for object detection.
  Input words:         10
  HLLSet bits (all):   28
  After gate ∩:        7
  Bits filtered:       21
  Output tokens:       7
  Materialized:        ['image', 'object', 'the', 'processes', 'neural', 'detection', 'data']
  LUT after 1 doc:     28
  Jaccard:             0.700
  Lost (filtered):     ['blockchain', 'for', 'quantum']
  False positives:     []


---
## 5. Multi-Document Learning

LUT accumulates TF from **all tokens** (pre-gate) across documents.
Frequent words dominate; noise fades. Converges after ~50-100 docs.

In [6]:
corpus = [
    'The neural network model processes image data for object detection.',
    'Deep learning algorithms require training data for classification tasks.',
    'Neural networks have revolutionized computer vision and pattern recognition.',
    'The neural network model processes image data for object detection.',
    'Document analysis uses natural language processing for text extraction.',
    'Neural networks perform feature extraction from raw image data.',
    'The classification model uses deep learning for image recognition.',
    'Object detection requires neural network models trained on image data.',
]

filt2 = HLLSetFilter(max_tokens=4096)
filt2.gate_hllset = gate_hllset

for i, text in enumerate(corpus):
    r = filt2.process(text)
    c = r.compare_to(text)
    print(f'Doc {i}: in={r.stats.input_tokens:2d} hll={r.stats.hllset_popcount:2d} '
          f'gate={r.stats.gate_popcount:2d} out={r.stats.output_tokens:2d} '
          f'jac={c.jaccard:.2f} LUT={r.lut_size:3d}')

Doc 0: in=10 hll=28 gate= 9 out= 9 jac=0.90 LUT= 28
Doc 1: in= 9 hll=25 gate= 6 out= 6 jac=0.50 LUT= 50
Doc 2: in= 9 hll=25 gate= 6 out= 6 jac=0.67 LUT= 74
Doc 3: in=10 hll=28 gate= 9 out= 9 jac=0.90 LUT= 74
Doc 4: in= 9 hll=25 gate= 7 out= 7 jac=0.78 LUT= 98
Doc 5: in= 9 hll=25 gate= 5 out= 5 jac=0.56 LUT=116
Doc 6: in= 9 hll=25 gate= 7 out= 7 jac=0.78 LUT=131
Doc 7: in= 9 hll=25 gate= 6 out= 6 jac=0.67 LUT=147


In [7]:
s = filt2.summary()
print(f'Documents:     {s["documents"]}')
print(f'LUT tokens:    {s["lut_size"]}')
print(f'LUT positions: {s["lut_positions"]}')
print(f'Gate popcount: {s["gate_popcount"]}')
print(f'Avg input:     {s["avg_input_tokens"]:.1f}')
print(f'Avg HLLSet:    {s["avg_hllset_popcount"]:.1f}')
print(f'Avg gate:      {s["avg_gate_popcount"]:.1f}')
print(f'Avg output:    {s["avg_output_tokens"]:.1f}')
print(f'Avg roundtrip: {s["avg_roundtrip_match"]:.3f}')

print(f'\nTop 10 TF: {filt2.lut.ranked_tokens()[:10]}')

Documents:     8
LUT tokens:    147
LUT positions: 144
Gate popcount: 32
Avg input:     9.2
Avg HLLSet:    25.8
Avg gate:      6.9
Avg output:    6.9
Avg roundtrip: 0.979

Top 10 TF: [('data', 5), ('neural', 5), ('image', 5), ('for', 5), ('image::data', 4), ('object', 3), ('data::for', 3), ('network', 3), ('model', 3), ('the', 3)]


---
## 6. Cross-Document Similarity — BSS Matrix

Related documents have higher BSS than unrelated — foundation for
shadow indexing (similar documents cluster by structure).

In [8]:
docs = {
    'ml_basics':   'Machine learning is a subset of artificial intelligence. '
                   'It enables systems to learn from data without explicit programming.',
    'ml_advanced': 'Deep learning uses neural networks with many layers. '
                   'It has revolutionized computer vision and natural language processing.',
    'cooking':     'The recipe calls for two cups of flour and three eggs. '
                   'Bake at 350 degrees for 30 minutes until golden brown.',
    'gardening':   'Plant tomatoes in full sun with well-drained soil. '
                  'Water deeply once per week during the growing season.',
}

hllsets = {}
for name, text in docs.items():
    t = DocumentTokenizer(max_tokens=4096).tokenize(text)
    hllsets[name] = hllset_py.HLLSet.from_tokens(t)

print(f'{"":15s}', end='')
for n in docs: print(f'{n:14s}', end='')
print()
for n1 in docs:
    print(f'{n1:15s}', end='')
    for n2 in docs:
        print(f'{hllsets[n1].bss_inclusion(hllsets[n2]):.4f}         ', end='')
    print()

ml_ml = hllsets['ml_basics'].bss_inclusion(hllsets['ml_advanced'])
ml_ck = hllsets['ml_basics'].bss_inclusion(hllsets['cooking'])
print(f'\nML→ML: {ml_ml:.4f}  |  ML→cooking: {ml_ck:.4f}  |  related>unrelated: {ml_ml > ml_ck}')

               ml_basics     ml_advanced   cooking       gardening     
ml_basics      1.0000         0.0435         0.0000         0.0204         
ml_advanced    0.0526         1.0000         0.0769         0.0408         
cooking        0.0000         0.0870         1.0000         0.0816         
gardening      0.0263         0.0435         0.0769         1.0000         

ML→ML: 0.0435  |  ML→cooking: 0.0000  |  related>unrelated: True


---
## 7. OCRPipeline — Gate + Filter + BPE Encode

Black-box interface: OCR text in, BPE token IDs out.

In [9]:
# Simulate tokenizer.json (fake BPE IDs for demo)
sim_vocab = {}
for i, w in enumerate(bpe_vocab):
    sim_vocab[w] = i + 1000
    sim_vocab[f'\u0120{w}'] = i + 5000

pipe = OCRPipeline()
pipe._bpe_vocab = sim_vocab
pipe._id_to_token = {v: k for k, v in sim_vocab.items()}

gi = pipe.set_gate()
print(f'Gate: {gi.valid_words} words from {gi.vocab_size} vocab, popcount={gi.gate_popcount}')

r = pipe.process('The neural quantum network processes image data for object detection.')
print(f'Compressed: {r.compressed_tokens}')
print(f'Token IDs:  {r.token_ids}')

Gate: 32 words from 64 vocab, popcount=32
Compressed: ['image', 'network', 'object', 'the', 'processes', 'neural', 'network::processes', 'detection', 'data']
Token IDs:  [1012, 1018, 1020, 1028, 1022, 1019, -1, 1006, 1004]


---
## 8. Latent Vocabulary — TF Survives Gate Changes

TF stored pre-gate → survives vocabulary expansion.
Gate controls what's rankable, not what's storable. (STANDARD.md §3.1)

In [10]:
# Phase 1: narrow gate
narrow = ['neural','network','model','image','data','detection','deep','learning','the','for']
filt3 = HLLSetFilter(max_tokens=4096)
filt3.gate_hllset = hllset_py.HLLSet.from_tokens(narrow)

for t in corpus[:4]:
    filt3.process(t)

tf_before = filt3.lut.tf('classification')
print(f'Narrow gate ({len(narrow)} words): TF("classification")={tf_before} (in LUT, filtered)')

# Phase 2: expand gate
wide = sorted(set(narrow + ['classification','revolutionized','recognition','extraction','training']))
filt3.gate_hllset = hllset_py.HLLSet.from_tokens(wide)

r = filt3.process('Deep learning algorithms require training data for classification tasks.')
tf_after = filt3.lut.tf('classification')
# assert tf_after == tf_before
print(f'Wide gate ({len(wide)} words): TF("classification")={tf_after} (same LUT, now survives)')
print(f'Materialized: {r.tokens}')
print(f'Latent vocabulary → instant activation on gate expansion')

Narrow gate (10 words): TF("classification")=1 (in LUT, filtered)
Wide gate (15 words): TF("classification")=2 (same LUT, now survives)
Materialized: ['classification', 'training', 'data', 'learning', 'deep', 'for']
Latent vocabulary → instant activation on gate expansion


---
## Summary

| Test | Result |
|------|--------|
| Tokenization | Words + bigrams + trigrams + sentence hashes |
| HLLSet | IICA-compliant, content-addressed, 32,768-bit |
| gate_TF HLLSet | Immutable, intersection-based probabilistic filter |
| Materialization | TF-ranked from persistent monotonic LUT |
| Multi-document | TF accumulates across docs, converges |
| Cross-document BSS | Related > unrelated |
| OCRPipeline | Gate → filter → BPE encode |
| Latent vocabulary | TF stored pre-gate, survives gate expansion |
| 3-gram encoding | False positive ~10⁻⁵ (effectively deterministic) |

** Direct hllset-py → hllset-core binding.
Reference implementation per hllset-next STANDARD.md.